# Before you begin


1.   Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.
2.   [Enable billing](https://support.google.com/cloud/answer/6293499#enable-billing) for the project.
3.   [Enable BigQuery](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) APIs for the project.


### Provide your credentials to the runtime

In [0]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


## Optional: Enable data table display

Colab includes the ``google.colab.data_table`` package that can be used to display large pandas dataframes as an interactive data table.
It can be enabled with:

In [0]:
%load_ext google.colab.data_table

If you would prefer to return to the classic Pandas dataframe display, you can disable this by running:
```python
%unload_ext google.colab.data_table
```

In [0]:
project_id = 'unique-apricot-258711'

In [0]:
%env GCLOUD_PROJECT=unique-apricot-258711

env: GCLOUD_PROJECT=unique-apricot-258711


In [0]:
import pandas as pd
import time

from google.cloud import bigquery


class BigQueryHelper(object):
    """
    Helper class to simplify common BigQuery tasks like executing queries,
    showing table schemas, etc without worrying about table or dataset pointers.

    See the BigQuery docs for details of the steps this class lets you skip:
    https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/reference.html
    """

    def __init__(self, active_project, dataset_name, max_wait_seconds=180):
        self.project_name = active_project
        self.dataset_name = dataset_name
        self.max_wait_seconds = max_wait_seconds
        self.client = bigquery.Client()
        self.__dataset_ref = self.client.dataset(self.dataset_name, project=self.project_name)
        self.dataset = None
        self.tables = dict()  # {table name (str): table object}
        self.__table_refs = dict()  # {table name (str): table reference}
        self.total_gb_used_net_cache = 0
        self.BYTES_PER_GB = 2**30

    def __fetch_dataset(self):
        """
        Lazy loading of dataset. For example,
        if the user only calls `self.query_to_pandas` then the
        dataset never has to be fetched.
        """
        if self.dataset is None:
            self.dataset = self.client.get_dataset(self.__dataset_ref)

    def __fetch_table(self, table_name):
        """
        Lazy loading of table
        """
        self.__fetch_dataset()
        if table_name not in self.__table_refs:
            self.__table_refs[table_name] = self.dataset.table(table_name)
        if table_name not in self.tables:
            self.tables[table_name] = self.client.get_table(self.__table_refs[table_name])

    def __handle_record_field(self, row, schema_details, top_level_name=''):
        """
        Unpack a single row, including any nested fields.
        """
        name = row['name']
        if top_level_name != '':
            name = top_level_name + '.' + name
        schema_details.append([{
            'name': name,
            'type': row['type'],
            'mode': row['mode'],
            'fields': pd.np.nan,
            'description': row['description']
                               }])
        # float check is to dodge row['fields'] == np.nan
        if type(row.get('fields', 0.0)) == float:
            return None
        for entry in row['fields']:
            self.__handle_record_field(entry, schema_details, name)

    def __unpack_all_schema_fields(self, schema):
        """
        Unrolls nested schemas. Returns dataframe with one row per field,
        and the field names in the format accepted by the API.
        Results will look similar to the website schema, such as:
            https://bigquery.cloud.google.com/table/bigquery-public-data:github_repos.commits?pli=1

        Args:
            schema: DataFrame derived from api repr of raw table.schema
        Returns:
            Dataframe of the unrolled schema.
        """
        schema_details = []
        schema.apply(lambda row:
            self.__handle_record_field(row, schema_details), axis=1)
        result = pd.concat([pd.DataFrame.from_dict(x) for x in schema_details])
        result.reset_index(drop=True, inplace=True)
        del result['fields']
        return result

    def table_schema(self, table_name):
        """
        Get the schema for a specific table from a dataset.
        Unrolls nested field names into the format that can be copied
        directly into queries. For example, for the `github.commits` table,
        the this will return `committer.name`.

        This is a very different return signature than BigQuery's table.schema.
        """
        self.__fetch_table(table_name)
        raw_schema = self.tables[table_name].schema
        schema = pd.DataFrame.from_dict([x.to_api_repr() for x in raw_schema])
        # the api_repr only has the fields column for tables with nested data
        if 'fields' in schema.columns:
            schema = self.__unpack_all_schema_fields(schema)
        # Set the column order
        schema = schema[['name', 'type', 'mode', 'description']]
        return schema

    def list_tables(self):
        """
        List the names of the tables in a dataset
        """
        self.__fetch_dataset()
        return([x.table_id for x in self.client.list_tables(self.dataset)])

    def estimate_query_size(self, query):
        """
        Estimate gigabytes scanned by query.
        Does not consider if there is a cached query table.
        See https://cloud.google.com/bigquery/docs/reference/rest/v2/jobs#configuration.dryRun
        """
        my_job_config = bigquery.job.QueryJobConfig()
        my_job_config.dry_run = True
        my_job = self.client.query(query, job_config=my_job_config)
        return my_job.total_bytes_processed / self.BYTES_PER_GB

    def query_to_pandas(self, query):
        """
        Execute a SQL query & return a pandas dataframe
        """
        my_job = self.client.query(query)
        start_time = time.time()
        while not my_job.done():
            if (time.time() - start_time) > self.max_wait_seconds:
                print("Max wait time elapsed, query cancelled.")
                self.client.cancel_job(my_job.job_id)
                return None
            time.sleep(0.1)
        # Queries that hit errors will return an exception type.
        # Those exceptions don't get raised until we call my_job.to_dataframe()
        # In that case, my_job.total_bytes_billed can be called but is None
        if my_job.total_bytes_billed:
            self.total_gb_used_net_cache += my_job.total_bytes_billed / self.BYTES_PER_GB
        return my_job.to_dataframe()

    def query_to_pandas_safe(self, query, max_gb_scanned=1):
        """
        Execute a query, but only if the query would scan less than `max_gb_scanned` of data.
        """
        query_size = self.estimate_query_size(query)
        if query_size <= max_gb_scanned:
            return self.query_to_pandas(query)
        msg = "Query cancelled; estimated size of {0} exceeds limit of {1} GB"
        print(msg.format(query_size, max_gb_scanned))

    def head(self, table_name, num_rows=5, start_index=None, selected_columns=None):
        """
        Get the first n rows of a table as a DataFrame.
        Does not perform a full table scan; should use a trivial amount of data as long as n is small.
        """
        self.__fetch_table(table_name)
        active_table = self.tables[table_name]
        schema_subset = None
        if selected_columns:
            schema_subset = [col for col in active_table.schema if col.name in selected_columns]
        results = self.client.list_rows(active_table, selected_fields=schema_subset,
            max_results=num_rows, start_index=start_index)
        results = [x for x in results]
        return pd.DataFrame(
            data=[list(x.values()) for x in results], columns=list(results[0].keys()))

In [0]:
bq_assistant = BigQueryHelper("bigquery-public-data", "github_repos")

In [0]:
QUERY = """
        SELECT
     reference_name,
     c.name,
     CONCAT(reference_bases, '->', alternate_bases[ORDINAL(1)].alt) AS mutation
   FROM
     `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` AS v, UNNEST(v.call) AS c
   WHERE
     # Only include biallelic SNPs.
     reference_bases IN ('A','C','G','T')
     AND alternate_bases[ORDINAL(1)].alt IN ('A','C','G','T')
     AND (ARRAY_LENGTH(alternate_bases) = 1
       OR (ARRAY_LENGTH(alternate_bases) = 2 AND alternate_bases[ORDINAL(2)].alt = '<*>'))
     # Skip homozygous reference calls and no-calls.
     AND EXISTS (SELECT g FROM UNNEST(c.genotype) AS g WHERE g > 0)
     AND NOT EXISTS (SELECT g FROM UNNEST(c.genotype) AS g WHERE g < 0)
     # Include only high quality calls.
     AND NOT EXISTS (SELECT ft FROM UNNEST(c.filter) ft WHERE ft NOT IN ('PASS', '.'))
        """

In [0]:
bq_assistant.estimate_query_size(QUERY)

5.950535892508924

###Task 1

In [0]:
client = bigquery.Client()

####Counting total rows in the table

In [0]:
#standardSQL

query = """
    SELECT 
      COUNT(1) AS number_of_rows
    FROM 
      `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823`
"""

query_job = client.query(
    query,
    location="US",
)  # API request - starts the query

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

,number_of_rows
0,105923159


####Counting variant calls in the table

Summing the lengths of call arrays

In [0]:
#standardSQL

query = """
    SELECT SUM(ARRAY_LENGTH(call)) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823`
"""

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

0.0


,number_of_calls
0,182104652


JOINing each row

In [0]:
#standardSQL
query = """
    SELECT COUNT(call) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
"""

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

0.0


,number_of_calls
0,182104652


Counting name in a call column

In [0]:
#standardSQL

query = """
    SELECT COUNT(call.name) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
"""

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

1.5263835601508617


,number_of_calls
0,182104652


####Counting variant and non-variant segments

In [0]:
#variant

query = """
    SELECT COUNT(1) AS number_of_real_variants
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call call
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.alternate_bases) AS alt WHERE alt.alt NOT IN ("<NON_REF>", "<*>"))
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

0.5340448450297117


,number_of_real_variants
0,38549388


In [0]:
#non variant

query = """
    SELECT COUNT(1) AS number_of_non_variants
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call call
    WHERE NOT EXISTS (SELECT 1 FROM UNNEST(v.alternate_bases) AS alt WHERE alt.alt NOT IN ("<NON_REF>", "<*>"))
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

0.5340448450297117


,number_of_non_variants
0,143555264


####Counting the variants called by each sample

In [0]:
#standardSQL
query = """
    SELECT call.name AS call_name, COUNT(call.name) AS call_count_for_call_set
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    GROUP BY call_name
    ORDER BY call_name 
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


1.5263835601508617


,call_name,call_count_for_call_set
0,NA12877,31592135
1,NA12878,28012646
2,NA12889,31028550
3,NA12890,30636087
4,NA12891,33487348
5,NA12892,27347886


In [0]:
#standardSQL

query = """
    SELECT call.name AS call_name, COUNT(call.name) AS call_count_for_call_set
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.alternate_bases) AS alt WHERE alt.alt NOT IN ("<NON_REF>", "<*>"))
    GROUP BY call_name
    ORDER BY call_name  
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

2.0604284051805735


,call_name,call_count_for_call_set
0,NA12877,6284275
1,NA12878,6397315
2,NA12889,6407532
3,NA12890,6448600
4,NA12891,6516669
5,NA12892,6494997


Filtering true variants by genotype

In [0]:
#standardSQL
query = """
    SELECT call.name AS call_name, COUNT(call.name) AS call_count_for_call_set
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    WHERE EXISTS (SELECT 1 FROM UNNEST(call.genotype) AS gt WHERE gt > 0) AND NOT EXISTS (SELECT 1 FROM UNNEST(call.genotype) AS gt WHERE gt < 0)
    GROUP BY call_name
    ORDER BY call_name  
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

#df = query_job.to_dataframe()
#df


4.239954333752394


####Counting samples in the table

In [0]:
#standardSQL
query = """
    SELECT COUNT(DISTINCT call.name) AS number_of_callsets
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


1.5263835601508617


,number_of_callsets
0,6


####Counting variants per chromosome

In [0]:
#standardSQL
query = """
    SELECT reference_name, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY reference_name
    ORDER BY
      CASE 
        WHEN SAFE_CAST(REGEXP_REPLACE(reference_name, '^chr', '') AS INT64) < 10 
          THEN CONCAT('0', REGEXP_REPLACE(reference_name, '^chr', '')) 
          ELSE REGEXP_REPLACE(reference_name, '^chr', '') 
      END 
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


3.341447635553777


,reference_name,number_of_variant_rows
0,chr1,615000
1,chr2,646401
2,chr3,542315
3,chr4,578600
4,chr5,496202
5,chr6,512152
6,chr7,459506
7,chr8,416376
8,chr9,344985
9,chr10,396773


####Counting high-quality variants per sample

Querying calls with multiple FILTER values

In [0]:
#standardSQL
query = """
    SELECT call_filter, COUNT(call_filter) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call, UNNEST(call.FILTER) AS call_filter
    GROUP BY call_filter
    ORDER BY number_of_calls
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

0.24804932065308094


,call_filter,number_of_calls
0,RefCall,11681534
1,PASS,26867854


FILTERing for high quality variant calls

In [0]:
#standardSQL
query = """
    SELECT reference_name, start_position, end_position, reference_bases, call.name AS call_name, (SELECT STRING_AGG(call_filter) FROM UNNEST(call.FILTER) AS call_filter) AS filters, ARRAY_LENGTH(call.FILTER) AS filter_count
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    WHERE EXISTS (SELECT 1 FROM UNNEST(call.FILTER) AS call_filter WHERE call_filter = 'PASS') AND ARRAY_LENGTH(call.FILTER) > 1
    ORDER BY filter_count DESC, reference_name, start_position, end_position, reference_bases, call_name
    LIMIT 10
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

4.281298340298235


,reference_name,start_position,end_position,reference_bases,call_name,filters,filter_count


Counting all high quality calls for each sample

In [0]:
#standardSQL
query = """
    SELECT call.name AS call_name, COUNT(1) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    WHERE NOT EXISTS (SELECT 1 FROM UNNEST(call.FILTER) AS call_filter WHERE call_filter != 'PASS')
    GROUP BY call_name
    ORDER BY call_name
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

1.7744328808039427


,call_name,number_of_calls
0,NA12877,29795946
1,NA12878,26118774
2,NA12889,29044992
3,NA12890,28717437
4,NA12891,31395995
5,NA12892,25349974


Counting all high quality true variant calls for each sample

In [0]:
#standardSQL
query = """
    SELECT call.name AS call_name, COUNT(1) AS number_of_calls
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v, v.call
    WHERE NOT EXISTS (SELECT 1 FROM UNNEST(call.FILTER) AS call_filter WHERE call_filter != 'PASS') AND EXISTS (SELECT 1 FROM UNNEST(call.genotype) as gt WHERE gt > 0)
    GROUP BY call_name
    ORDER BY call_name
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

4.488003654405475


,call_name,number_of_calls
0,NA12877,4486610
1,NA12878,4502017
2,NA12889,4422706
3,NA12890,4528725
4,NA12891,4424094
5,NA12892,4495753


####Condensing queries

In [0]:
#standardSQL
query = """
    SELECT reference_name, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call WHERE EXISTS (SELECT 1 FROM UNNEST(call.genotype) AS gt WHERE gt > 0))
    GROUP BY reference_name
    ORDER BY reference_name
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

3.341447635553777


,reference_name,number_of_variant_rows
0,chr1,615000
1,chr10,396773
2,chr11,391260
3,chr12,382841
4,chr13,298044
5,chr14,258756
6,chr15,234569
7,chr16,247671
8,chr17,224403
9,chr18,227200


In [0]:
#standardSQL
query = """
    SELECT reference_name, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY reference_name
    ORDER BY reference_name
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

3.341447635553777


,reference_name,number_of_variant_rows
0,chr1,615000
1,chr10,396773
2,chr11,391260
3,chr12,382841
4,chr13,298044
5,chr14,258756
6,chr15,234569
7,chr16,247671
8,chr17,224403
9,chr18,227200


In [0]:
#standardSQL
query = """
    SELECT REGEXP_REPLACE(reference_name, '^chr', '') AS chromosome, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY chromosome
    ORDER BY chromosome
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


3.341447635553777


,chromosome,number_of_variant_rows
0,1,615000
1,10,396773
2,11,391260
3,12,382841
4,13,298044
5,14,258756
6,15,234569
7,16,247671
8,17,224403
9,18,227200


In [0]:
#standardSQL
#the result will be error because X and Y cannot convert to integer
query = """
    SELECT CAST(REGEXP_REPLACE(reference_name, '^chr', '') AS INT64) AS chromosome, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY chromosome
    ORDER BY chromosome
    """

query_job = client.query(query)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df

3.341447635553777


BadRequest: ignored

In [0]:
#standardSQL
query = """
    SELECT CASE WHEN SAFE_CAST(REGEXP_REPLACE(reference_name, '^chr', '') AS INT64) < 10 THEN CONCAT('0', REGEXP_REPLACE(reference_name, '^chr', '')) ELSE REGEXP_REPLACE(reference_name, '^chr', '') END AS chromosome, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY chromosome
    ORDER BY chromosome
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


3.341447635553777


,chromosome,number_of_variant_rows
0,01,615000
1,02,646401
2,03,542315
3,04,578600
4,05,496202
5,06,512152
6,07,459506
7,08,416376
8,09,344985
9,10,396773


In [0]:
#standardSQL
query = """
    SELECT reference_name, COUNT(reference_name) AS number_of_variant_rows
    FROM `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
    WHERE EXISTS (SELECT 1 FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt WHERE gt > 0)
    GROUP BY reference_name
    ORDER BY 
      CASE
        WHEN SAFE_CAST(REGEXP_REPLACE(reference_name, '^chr', '') AS INT64) < 10 
          THEN CONCAT('0', REGEXP_REPLACE(reference_name, '^chr', '')) 
          ELSE REGEXP_REPLACE(reference_name, '^chr', '') 
      END
    """

query_job = client.query(
    query,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query))

df = query_job.to_dataframe()
df


3.341447635553777


,reference_name,number_of_variant_rows
0,chr1,615000
1,chr2,646401
2,chr3,542315
3,chr4,578600
4,chr5,496202
5,chr6,512152
6,chr7,459506
7,chr8,416376
8,chr9,344985
9,chr10,396773


####Writing user-defined functions

In [0]:
#standardSQL
query_f = """
CREATE TEMPORARY FUNCTION SortableChromosome(reference_name STRING)
  RETURNS STRING AS (
  -- Remove the leading "chr" (if any) in the reference_name
  -- If the chromosome is 1 - 9, prepend a "0" since
  -- "2" sorts after "10", but "02" sorts before "10".
  CASE
    WHEN SAFE_CAST(REGEXP_REPLACE(reference_name, '^chr', '') AS INT64) < 10
      THEN CONCAT('0', REGEXP_REPLACE(reference_name, '^chr', ''))
      ELSE REGEXP_REPLACE(reference_name, '^chr', '')
  END
);

SELECT
  reference_name,
  COUNT(reference_name) AS number_of_variant_rows
FROM
  `bigquery-public-data.human_genome_variants.platinum_genomes_deepvariant_variants_20180823` v
WHERE
  EXISTS (SELECT 1
            FROM UNNEST(v.call) AS call, UNNEST(call.genotype) AS gt
          WHERE gt > 0)
GROUP BY
  reference_name
ORDER BY SortableChromosome(reference_name)
"""

query_f_job = client.query(
    query_f,
    # Location must match that of the dataset(s) referenced in the query.
    location="US",
)

print(bq_assistant.estimate_query_size(query_f))

dff = query_f_job.to_dataframe()
dff

3.341447635553777


,reference_name,number_of_variant_rows
0,chr1,615000
1,chr2,646401
2,chr3,542315
3,chr4,578600
4,chr5,496202
5,chr6,512152
6,chr7,459506
7,chr8,416376
8,chr9,344985
9,chr10,396773
